# Healthy Start — Python Detection Server (Colab)

Runs the FastAPI dental detection server on Colab with a public tunnel.
The public URL connects to the Vercel frontend at `levelupwrestlingapp.com/hs/loop`.

**Requirements:** GPU runtime (T4 recommended). Go to Runtime > Change runtime type > T4 GPU.

## 1. Clone Repo & Install Dependencies

In [ ]:
# Clone the repo
!git clone https://github.com/sportsmockery/LevelUp.git /content/levelup
%cd /content/levelup/python

# Install dependencies
!pip install -q ultralytics>=8.3.0 fastapi>=0.115.0 uvicorn>=0.34.0 \
  python-multipart>=0.0.18 Pillow>=11.0.0 \
  pyngrok torch torchvision opencv-python-headless

# segment-anything is not on PyPI — install from Meta's GitHub
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

print('\n--- Dependencies installed ---')

## 2. Download Models

**SAM model** downloads automatically from Meta.  
**YOLO model** — upload `yolov12s_010826.pt` using the file panel on the left,  
or place it in Google Drive and mount below.

In [ ]:
import os, urllib.request, pathlib

MODEL_DIR = pathlib.Path('/content/levelup/python')

# --- SAM vit_b (358 MB) ---
SAM_URL = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'
SAM_PATH = MODEL_DIR / 'sam_vit_b_01ec64.pth'

if not SAM_PATH.exists():
    print('Downloading SAM vit_b model (358 MB)...')
    urllib.request.urlretrieve(SAM_URL, str(SAM_PATH))
    print(f'SAM model saved: {SAM_PATH} ({SAM_PATH.stat().st_size / 1e6:.0f} MB)')
else:
    print(f'SAM model already exists: {SAM_PATH}')

# --- YOLO model (18 MB) ---
YOLO_PATH = MODEL_DIR / 'yolov12s_010826.pt'

if not YOLO_PATH.exists():
    # Try Google Drive mount
    gdrive_path = pathlib.Path('/content/drive/MyDrive/models/yolov12s_010826.pt')
    if gdrive_path.exists():
        import shutil
        shutil.copy(str(gdrive_path), str(YOLO_PATH))
        print(f'YOLO model copied from Google Drive: {YOLO_PATH}')
    else:
        print('\n' + '='*60)
        print('ACTION REQUIRED: Upload yolov12s_010826.pt')
        print('='*60)
        print('Options:')
        print('  1. Drag yolov12s_010826.pt into the Files panel (left sidebar)')
        print('     Then run: !cp /content/yolov12s_010826.pt /content/levelup/python/')
        print('  2. Mount Google Drive and place it at:')
        print(f'     {gdrive_path}')
        print('='*60 + '\n')
else:
    print(f'YOLO model already exists: {YOLO_PATH}')

# Verify
print(f'\nSAM ready: {SAM_PATH.exists()} ({SAM_PATH.stat().st_size / 1e6:.0f} MB)' if SAM_PATH.exists() else 'SAM: MISSING')
print(f'YOLO ready: {YOLO_PATH.exists()} ({YOLO_PATH.stat().st_size / 1e6:.0f} MB)' if YOLO_PATH.exists() else 'YOLO: MISSING')

## 2b. (Optional) Upload YOLO model manually

If the YOLO model wasn't found above, upload it with the Files panel then run this cell:

In [ ]:
# Run this after uploading yolov12s_010826.pt to /content/
import shutil, pathlib
src = pathlib.Path('/content/yolov12s_010826.pt')
dst = pathlib.Path('/content/levelup/python/yolov12s_010826.pt')
if src.exists() and not dst.exists():
    shutil.copy(str(src), str(dst))
    print(f'Copied to {dst}')
elif dst.exists():
    print('YOLO model already in place')
else:
    print('Upload yolov12s_010826.pt to /content/ first')

## 3. API Keys & Training Data

Set your Roboflow API key to auto-fetch dental datasets.  
Without this, the loop needs manually uploaded images.

In [ ]:
import os, pathlib

# --- Set API key (required for auto-fetch) ---
ROBOFLOW_API_KEY = ''  # <-- PASTE YOUR ROBOFLOW KEY HERE

if ROBOFLOW_API_KEY:
    os.environ['ROBOFLOW_API_KEY'] = ROBOFLOW_API_KEY
    print(f'ROBOFLOW_API_KEY set ({ROBOFLOW_API_KEY[:6]}...)')

    # Write .env.local so the server process picks it up at startup
    env_path = pathlib.Path('/content/levelup/.env.local')
    env_path.write_text(f'ROBOFLOW_API_KEY={ROBOFLOW_API_KEY}\n')
    print(f'Written to {env_path}')
else:
    print('WARNING: No ROBOFLOW_API_KEY set.')
    print('The loop will fail unless you manually upload images.\n')

# Create directories the server/loop expects
for d in ['data/raw/train/images', 'data/labeling_queue', 'data/truth_engine', 'data/audit_history', 'runs']:
    pathlib.Path(f'/content/levelup/python/{d}').mkdir(parents=True, exist_ok=True)
print('Data directories created')

# --- Pre-fetch datasets if key is available ---
if ROBOFLOW_API_KEY:
    print('\nFetching dental datasets from Roboflow...')
    os.environ['PYTHONPATH'] = '/content/levelup/python'
    import sys
    sys.path.insert(0, '/content/levelup/python')
    from broken_contacts.dataset_fetcher import fetch_all_datasets
    results = fetch_all_datasets(
        api_key=ROBOFLOW_API_KEY,
        target_dir='data/raw/train/images',
        skip_downloaded=True,
    )
    total = sum(r.get('images_copied', 0) for r in results)
    print(f'\nFetched {total} images across {len(results)} datasets')
    for r in results:
        print(f"  {r.get('dataset', '?')}: {r.get('images_copied', 0)} images")

# Count available images
img_dir = pathlib.Path('/content/levelup/python/data/raw/train/images')
img_count = len([f for f in img_dir.rglob('*') if f.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}])
print(f'\nTotal training images available: {img_count}')
if img_count == 0:
    print('WARNING: No images! Upload .jpg/.png files to data/raw/train/images/')

## 4. Start Public Tunnel (ngrok)

Get a free ngrok auth token at https://dashboard.ngrok.com/get-started/your-authtoken  
Paste it below. This gives you a public HTTPS URL that Vercel can reach.

## 4. Train Models (Required Before Loop Gives Useful Results)

The loop needs two trained models to produce clinical diagnoses:

1. **9-class YOLO Detector** — finds tooth surfaces, gaps, restorations, hardware
2. **ResNet18 Contact Classifier** — classifies each contact as normal/open/unclear

Without these, the loop only runs the base YOLO (generic tooth detection) with no clinical output.

### Step 4a: Generate Synthetic Training Data
Creates labeled dental images with teeth, gaps, restorations, and gum lines.

In [ ]:
import sys, os
os.chdir('/content/levelup/python')
sys.path.insert(0, '/content/levelup/python')
os.environ['PYTHONPATH'] = '/content/levelup/python'

from broken_contacts.synth_generator import generate_dataset

# Generate 500 synthetic images (400 train / 100 val) with 9-class labels
# These have perfect annotations for: tooth_crown, mesial_surface, distal_surface,
# occlusal_surface, restoration_margin, contact_gap_candidate, gingival_margin
summary = generate_dataset(
    output_dir='data/synth_detector',
    num_images=500,
    train_split=0.8,
    img_w=640,
    img_h=480,
    gap_probability=0.3,          # 30% chance of gap between teeth
    restoration_probability=0.2,  # 20% chance of restoration per tooth
)

print(f"\ndata.yaml: {summary['data_yaml']}")
print(f"Ready for YOLO training!")

In [ ]:
NGROK_AUTH_TOKEN = ''  # <-- PASTE YOUR TOKEN HERE

from pyngrok import ngrok

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
else:
    print('WARNING: No ngrok token set. Get one free at https://dashboard.ngrok.com')
    print('Without a token, the tunnel may be rate-limited.\n')

# Open tunnel to port 8100
public_url = ngrok.connect(8100, 'http')
print('='*60)
print(f'PUBLIC URL: {public_url}')
print('='*60)
print(f'\nSet this in Vercel environment variables:')
print(f'  HS_DETECTION_URL = {public_url}')
print(f'\nOr run locally:')
print(f'  vercel env add HS_DETECTION_URL')
print(f'  (paste: {public_url})')

### Step 4b: Train the 9-Class YOLO Detector

Fine-tunes the base YOLO model on synthetic data. Takes ~15-20 min on T4 GPU.  
Output: `runs/detect/contacts_detector_v1/weights/best.pt`

In [ ]:
from ultralytics import YOLO

# Load base model and fine-tune on 9-class synthetic dental data
model = YOLO('/content/levelup/python/yolov12s_010826.pt')

results = model.train(
    data='/content/levelup/python/data/synth_detector/data.yaml',
    epochs=80,
    imgsz=640,
    batch=16,
    name='contacts_detector_v1',
    patience=20,
    lr0=0.0005,
    augment=True,
    save=True,
    save_period=10,
    val=True,
    plots=True,
    verbose=True,
    project='runs/detect',
)

# Validate
import pathlib
best = pathlib.Path('runs/detect/contacts_detector_v1/weights/best.pt')
if best.exists():
    print(f'\nDetector trained! Best model: {best}')
    val_model = YOLO(str(best))
    metrics = val_model.val(data='/content/levelup/python/data/synth_detector/data.yaml')
    print(f'mAP50:    {metrics.box.map50:.4f}')
    print(f'mAP50-95: {metrics.box.map:.4f}')
else:
    print('WARNING: best.pt not found — training may have failed')

### Step 4c: Generate Contact Crops for Classifier Training

Uses the newly trained detector to find contacts, then extracts crops.  
These are auto-sorted into normal/open/unclear based on gap presence in synthetic labels.

In [ ]:
import json, pathlib, shutil, random
from PIL import Image
from broken_contacts.config import Config
from broken_contacts.crops import generate_crops_from_labels

config = Config()
config.crop_size = 224

# Generate crops from synthetic data (which has perfect labels including gap info)
synth_dir = pathlib.Path('data/synth_detector')
classifier_dir = pathlib.Path('data/classifier_dataset')

for split in ['train', 'val']:
    for cls in ['normal_contact', 'open_contact', 'unclear_contact']:
        (classifier_dir / split / cls).mkdir(parents=True, exist_ok=True)

crop_count = {'normal_contact': 0, 'open_contact': 0, 'unclear_contact': 0}

for split in ['train', 'val']:
    img_dir = synth_dir / split / 'images'
    lbl_dir = synth_dir / split / 'labels'
    rel_dir = synth_dir / 'relational'

    if not img_dir.exists():
        continue

    for img_path in sorted(img_dir.glob('*.jpg')):
        label_path = lbl_dir / f'{img_path.stem}.txt'
        rel_path = rel_dir / f'{img_path.stem}.json'

        if not label_path.exists():
            continue

        # Read relational annotations to know which pairs have gaps
        gap_pairs = set()
        if rel_path.exists():
            with open(rel_path) as f:
                rel = json.load(f)
            for pair in rel.get('contact_pairs', []):
                if pair.get('has_gap', False):
                    gap_pairs.add(pair['pair_index'])

        # Generate crops
        try:
            crops = generate_crops_from_labels(
                image_path=str(img_path),
                label_path=str(label_path),
                config=config,
                output_dir=None,  # don't save yet
            )
        except Exception:
            continue

        for i, crop_data in enumerate(crops):
            crop_img = crop_data.get('crop_image')
            if crop_img is None:
                continue

            # Assign label based on gap presence
            if i in gap_pairs:
                label = 'open_contact'
            elif random.random() < 0.05:  # 5% unclear for training
                label = 'unclear_contact'
            else:
                label = 'normal_contact'

            out_path = classifier_dir / split / label / f'{img_path.stem}_pair{i}.jpg'
            crop_img.save(str(out_path))
            crop_count[label] += 1

print('Classifier dataset generated:')
for cls, count in crop_count.items():
    print(f'  {cls}: {count}')

total = sum(crop_count.values())
print(f'  Total: {total} crops')

if total < 50:
    print('\nWARNING: Very few crops. Classifier training may underperform.')
    print('Consider generating more synthetic images (increase num_images in Step 4a).')

### Step 4d: Train the Contact Classifier (ResNet18)

Trains the 3-class classifier on contact crops. Takes ~5 min on T4 GPU.  
Output: `runs/classify/contacts_classifier_v1/best.pt`

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from broken_contacts.classifier import build_classifier_model
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Transforms
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder('data/classifier_dataset/train', transform=train_tf)
val_ds = datasets.ImageFolder('data/classifier_dataset/val', transform=val_tf)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Classes: {train_ds.classes}')

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

# Build model
model = build_classifier_model(num_classes=len(train_ds.classes), pretrained=True)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

run_dir = Path('runs/classify/contacts_classifier_v1')
run_dir.mkdir(parents=True, exist_ok=True)
best_val_acc = 0.0

EPOCHS = 30
for epoch in range(EPOCHS):
    # Train
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        train_correct += (predicted == labels).sum().item()
        train_total += labels.size(0)
    scheduler.step()

    # Validate
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    train_acc = train_correct / max(train_total, 1)
    val_acc = val_correct / max(val_total, 1)

    if (epoch + 1) % 5 == 0 or val_acc > best_val_acc:
        print(f'Epoch {epoch+1}/{EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'val_acc': val_acc,
            'classes': train_ds.classes,
        }, str(run_dir / 'best.pt'))

# Save final
torch.save({
    'epoch': EPOCHS,
    'model_state_dict': model.state_dict(),
    'val_acc': val_acc,
    'classes': train_ds.classes,
}, str(run_dir / 'last.pt'))

print(f'\nClassifier trained! Best val accuracy: {best_val_acc:.4f}')
print(f'Model saved: {run_dir / "best.pt"}')

### Step 4e: Verify Both Models Are Ready

In [ ]:
import pathlib

detector = pathlib.Path('runs/detect/contacts_detector_v1/weights/best.pt')
classifier = pathlib.Path('runs/classify/contacts_classifier_v1/best.pt')

print('Model Status:')
print(f'  Detector (9-class YOLO):  {"READY" if detector.exists() else "MISSING"}'
      + (f' ({detector.stat().st_size / 1e6:.1f} MB)' if detector.exists() else ''))
print(f'  Classifier (ResNet18):    {"READY" if classifier.exists() else "MISSING"}'
      + (f' ({classifier.stat().st_size / 1e6:.1f} MB)' if classifier.exists() else ''))

if detector.exists() and classifier.exists():
    print('\nBoth models ready! The loop will now produce:')
    print('  - Contact pair detection with gap measurements')
    print('  - Dual-layer labels (morphology + clinical)')
    print('  - Flagged contacts (food_trap_risk, restoration_failure)')
    print('  - Clinical diagnosis per patient')
    print('\nProceed to Step 5 (ngrok) and Step 6 (start server).')
else:
    print('\nWARNING: Missing models. Run the training cells above first.')

## 5. Run the Server

This cell blocks while the server runs. The tunnel URL above is live.  
Visit `levelupwrestlingapp.com/hs/loop` and click **Start**.

In [ ]:
import os

# Set env vars so the shell subprocess can find modules and keys
os.environ['PYTHONPATH'] = '/content/levelup/python'

print('Starting FastAPI server on port 8100...')
print('Tunnel active — Vercel frontend can now connect.\n')

# Verify models exist before starting
for name in ['yolov12s_010826.pt', 'sam_vit_b_01ec64.pth']:
    path = f'/content/levelup/python/{name}'
    if os.path.exists(path):
        print(f'  {name}: OK ({os.path.getsize(path) / 1e6:.0f} MB)')
    else:
        print(f'  {name}: MISSING — server will fail to load!')

# Count images
import pathlib
img_dir = pathlib.Path('/content/levelup/python/data/raw/train/images')
img_count = len([f for f in img_dir.rglob('*') if f.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}])
print(f'  Training images: {img_count}')
print(f'  ROBOFLOW_API_KEY: {"set" if os.environ.get("ROBOFLOW_API_KEY") else "NOT SET"}')
print()

!cd /content/levelup/python && python -m uvicorn server:app --host 0.0.0.0 --port 8100

## 6. (Optional) Test the Server

In [ ]:
# Run in a separate cell while the server is running
# (open a new code cell, the server cell above will keep running)
import requests
r = requests.get('http://localhost:8100/health')
print(r.json())